# PocketVina quickstart

<a href="https://colab.research.google.com/github/BIMSBbioinfo/PocketVina/blob/main/notebooks/quickstart_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[PocketVina](https://github.com/BIMSBbioinfo/PocketVina) docks the example ligands into the two example
proteins: P2Rank finds the pockets, the GPU docks every ligand into the five best pockets of each protein. Then the
results in 3D, one protein × ligand pair, its PoseBusters checks, and its poses as an SDF file.

On Colab: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

## Install

On Colab this installs Java, OpenCL and PocketVina (a few minutes; the steps are in `notebooks/colab_setup.py`).
Elsewhere it installs nothing: install PocketVina first (`pip install git+https://github.com/BIMSBbioinfo/PocketVina`).

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/BIMSBbioinfo/PocketVina"
REPO_REF = "main"  # the branch or tag to install
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/PocketVina"):
        subprocess.run(f"git clone -q --depth 1 -b {REPO_REF} {REPO_URL} /content/PocketVina", shell=True, check=True)
    sys.path.insert(0, "/content/PocketVina/notebooks")
import colab_setup  # notebooks/colab_setup.py

colab_setup.setup(REPO_URL, REPO_REF)

## Dock

The two example proteins and the nine example ligands. At the end the best score of each protein × ligand
appears (`pocketvina.best_scores()` gives the whole table).

In [ ]:
from pocketvina import PocketVinaGPU, PocketVinaConfig, P2RankConfig, PocketConfig

with open("proteins.txt", "w") as f:  # the proteins to dock, one structure file per line
    f.write("../example/proteins/1fbl.pdb\n../example/proteins/2W83.pdb\n")

config = PocketVinaConfig(
    protein_paths="proteins.txt",          # text file: one protein structure per line
    smiles_file="../example/ligands.txt",  # CSV file: molecule_name,smiles
    output_dir="results",                  # folder for the results
    output_format="atlas",                 # results as tables and pose files, needed for the cells below
    batch_size=10,                         # molecules per docking batch
    p2rank_config=P2RankConfig(            # pocket detection
        threads=8,                         # CPU threads
        visualizations=False,              # P2Rank's PyMOL files: not needed
        vis_copy_proteins=False,
        output_dir="results",              # the same folder as above
        model="default",                   # "default" for crystal structures, "alphafold" for AlphaFold models
    ),
    pocket_config=PocketConfig(            # the docking box around each pocket
        size_x=20, size_y=20, size_z=20,   # box size in Angstrom
        thread=8000,                       # GPU threads per docking (at least 1000)
    ),
)
pocketvina = PocketVinaGPU(config, htvs_mode=True)  # dock the 5 best pockets of each protein
pocketvina.execute_docking()  # prints how long P2Rank, receptor and docking took per protein, then the best scores

## Results docked before

To look at a results folder of an earlier run without docking again (e.g. after a kernel restart): run *Install*, then this cell (not *Dock*). It works for any results folder, also one docked on a cluster; the cells below work the same.

In [ ]:
from pocketvina import Results

pocketvina = Results("results")  # the results folder (output_dir) of a run
pocketvina.best_scores()

## All results in 3D

Like the protein page of the PocketVina Atlas: click a ligand in *Top hits*, a pocket chip or a mode; *Save image*
saves a PNG.

In [ ]:
pocketvina.show()

## One protein × ligand pair

`select` picks a pair by the ids in the table above; by default its best pose.

In [ ]:
pair = pocketvina.select(protein="1fbl", ligand="molecule1")
pair

The 3D page with this pair selected:

In [ ]:
pair.show()

Other poses: `pocket=2` and `mode=3` pick one, `mode="all"` every mode of the pocket, and `pocket="all"` the
best pose in every docked pocket:

In [ ]:
pair = pocketvina.select(protein="1fbl", ligand="molecule1", pocket="all")
pair.show()

## PoseBusters

Checks that the selected poses are physically plausible (bond lengths, clashes with the protein, ...): one row per pose. It checks exactly the poses you select, e.g.

- `pocketvina.select(protein="1fbl", ligand="molecule1").bust()`: the best pose
- `pocketvina.select(protein="1fbl", ligand="molecule1", pocket=1, mode=10).bust()`: mode 10 of pocket 1
- `pocketvina.select(protein="1fbl", ligand="molecule1", pocket=1, mode="all").bust()`: every mode of pocket 1

Here: the best pose of every pocket (the `pocket="all"` selection above).

In [ ]:
pair.bust()

## Save

The selected poses as one SDF file (`fmt="pdbqt"` for PDBQT); on Colab it is in `PocketVina/notebooks/` in the
file browser.

In [ ]:
pair.export()

## Next

- Your own proteins and ligands: a text file with one structure per line as `protein_paths`, a `molecule_name,smiles`
  CSV file as `smiles_file`. Other pockets than the 5 best: `top_pockets=7` or `pockets="10-12"` in the config
  with `PocketVinaGPU(config)` (without `htvs_mode`); neither of them: every pocket.
- Results docked before: the *Results docked before* cell above.
- The [documentation](https://bioinformatics.mdc-berlin.de/PocketVina/) has the other settings and the command line.
- P2Rank first, docking later (e.g. the pockets of a whole proteome once): `P2RankProcessor` writes the pockets,
  `PocketVinaConfig(pockets_dir=...)` docks into them without running P2Rank again; see the
  [quick start](https://bioinformatics.mdc-berlin.de/PocketVina/quickstart.html).